In [1]:
import re
import unicodedata
import pandas as pd

In [2]:
df = pd.read_csv(r"dataset\filtering\news_filtered.csv")
print(f'''
      Shape: {df.shape}
      Columns: {df.columns.tolist()}
''')
df.head()


      Shape: (202, 6)
      Columns: ['content_id', 'text', 'section', 'published_date', 'category', 'keywords']



,content_id,text,section,published_date,category,keywords
0,1808882,CelcomDigi has refreshed its Postpaid 5G plans...,Tech,2026-02-05 00:00:00,5G,"Telcos,5G,Internet,Technology,Smartphones"
1,1807093,Apple Inc.’s new second-generation AirTag is b...,Tech,2026-02-03 00:00:00,Gadgets,Gadgets
2,1801286,ATLANTA: Your watch says you had three hours o...,Tech,2026-01-26 00:00:00,Gadgets,"Gadgets,Wearables,Technology"
3,1796413,"TEMBISA, South Africa: Almost as soon as Ansel...",Tech,2026-01-19 00:00:00,Gadgets,"Energy,Gadgets,Technology,Environment,Africa"
4,1796384,"CASABLANCA, Morocco: Despite the availability ...",Tech,2026-01-19 00:00:00,Gadgets,"Gadgets,Technology"


## Text Cleaning

In [3]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = re.sub(r"''|``|‘’", '"', text)
    
    text = text.replace('\xad', '')
    
    # Links and HTML remove
    text = re.sub(r'\b(?:https?://|www\.)?\S+\.\S+(?:/\S*)?', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    
    # replace control chars with space instead of deleting
    text = ''.join(ch if unicodedata.category(ch)[0] != "C" else ' ' for ch in text)
    
    text = re.sub(r'([.!?])([A-Z])', r'\1 \2', text) # Fix glued sentences
    
    text = re.sub(r'\s+', ' ', text)
    
    text = re.sub(r'©.*?(?:\.|$)', '', text) # Ending year mention remove
    
    text = re.sub(r'\(\s+(?=[A-Za-z–-])', ', ', text) # Open parenthesis to commas
    
    text = re.sub(r'^[A-Z\s/]+(?:,\s*[A-Za-z\s]+)?:\s', '', text) # Location remove
    
    return text.strip()

In [4]:
df['clean_text'] = df['text'].apply(clean_text)

In [5]:
df['published_date'] = pd.to_datetime(df['published_date'])

In [6]:
df.head()

,content_id,text,section,published_date,category,keywords,clean_text
0,1808882,CelcomDigi has refreshed its Postpaid 5G plans...,Tech,2026-02-05,5G,"Telcos,5G,Internet,Technology,Smartphones",CelcomDigi has refreshed its Postpaid 5G plans...
1,1807093,Apple Inc.’s new second-generation AirTag is b...,Tech,2026-02-03,Gadgets,Gadgets,Apple new second-generation AirTag is basicall...
2,1801286,ATLANTA: Your watch says you had three hours o...,Tech,2026-01-26,Gadgets,"Gadgets,Wearables,Technology",Your watch says you had three hours of deep sl...
3,1796413,"TEMBISA, South Africa: Almost as soon as Ansel...",Tech,2026-01-19,Gadgets,"Energy,Gadgets,Technology,Environment,Africa",Almost as soon as Anselmo Munghabe plopped the...
4,1796384,"CASABLANCA, Morocco: Despite the availability ...",Tech,2026-01-19,Gadgets,"Gadgets,Technology","Despite the availability of haptic, or touch, ..."


In [7]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gaura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Sentence Splitting and Naration Fixes

In [8]:
import re

def merge_quote_splits(sentences):
    merged = []
    buffer = ""

    for sent in sentences:
        if buffer:
            buffer += " " + sent
            if buffer.count('"') % 2 == 0:
                merged.append(buffer.strip())
                buffer = ""
        else:
            if sent.count('"') % 2 != 0:
                buffer = sent
            else:
                merged.append(sent)

    if buffer:
        merged.append(buffer)

    return merged

In [9]:
def secondary_split(sent):
    return re.split(r'(?<!")\.\s+(?=[A-Z])', sent)

def refine_sentences(sent_list):
    refined = []
    for s in sent_list:
        refined.extend(secondary_split(s))
    return refined

In [10]:
def refine_sentences(sent_list):
    refined = []
    for s in sent_list:
        refined.extend(secondary_split(s))
    return refined

In [11]:
df['sentences'] = df['clean_text'].apply(
    lambda x: refine_sentences(
        merge_quote_splits(sent_tokenize(x))
    )
)

In [12]:
df_sent = df.explode('sentences').reset_index(drop=True)
df_sent.rename(columns={'sentences': 'sentence'}, inplace=True)

In [13]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(5412, 8)


Gadgets    4984
5G          428
Name: category, dtype: int64

In [14]:
df_sent = df_sent[df_sent['sentence'].str.len() > 40]

In [15]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(5120, 8)


Gadgets    4712
5G          408
Name: category, dtype: int64

## Removing publisher names from end of articles

In [16]:
boilerplate_pattern= r'[.|"]\s*[–-]\s*[A-Za-z\s]{1,40}$'

df_sent['sentence'] = df_sent['sentence'].str.replace(
    boilerplate_pattern,
    '',
    regex=True
)

In [17]:
df_final = df_sent[['content_id', 'sentence', 'published_date', 'category']]

print(df_final.shape)
df_final.head()

(5120, 4)


,content_id,sentence,published_date,category
0,1808882,CelcomDigi has refreshed its Postpaid 5G plans...,2026-02-05,5G
1,1808882,The company said the newest addition is the Po...,2026-02-05,5G
2,1808882,The plan also includes access to streaming ser...,2026-02-05,5G
3,1808882,Other unlimited data with uncapped speed plans...,2026-02-05,5G
4,1808882,The unlimited quota is subject to the company'...,2026-02-05,5G


In [18]:
df_final.to_csv(r"dataset\preprocessing\prePro-news_filtered.csv", index=False)